In [9]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import os
import torch
import pandas as pd
import sys
import matplotlib.pyplot as plt
import scipy
from pyproj import Transformer
sys.path.append('../')
from src.plotting import Plotting
from src.utilities import load_ase_datasets, reverse_standardize
from src.ice import temperature_to_conductivity, conductivity_to_attenu_rate

aux_data = load_ase_datasets()
xs_ase, ys_ase = aux_data['coords']['xs'], aux_data['coords']['ys']
print("Keys of the auxiliary data dictionary:", aux_data.keys())

  [load_ase_datasets] Loading reference coordinates …
  [load_ase_datasets] Loading Bedmap3 …
  [load_ase_datasets] Computing Tpmp …
  [load_ase_datasets] Computing surface slope …
  [load_ase_datasets] Loading velocity …
  [load_ase_datasets] Loading domain mask …
  [load_ase_datasets] Loading surface mass balance (SMB) …
  [load_ase_datasets] All datasets ready.
Keys of the auxiliary data dictionary: dict_keys(['coords', 'sdem', 'bed', 'H', 'Tpmp', 'slope_x', 'slope_y', 'slope_mag', 'vx', 'vy', 'vel', 'mask', 'domain_bound_line', 'smb'])


In [ ]:
# load the calibrated depth-averaged attenuation rate
calibrated_folder = "../data/final-obs-evidence/"
data_path       = os.getenv('RAW_DATA_PATH')
param_path      = os.getenv('PARAM_PATH')
Ns_mean_filename     = 'trainingAll_Ns_sim_mean_masked.mat'
Ns_std_filename      = 'trainingAll_Ns_sim_std_masked.mat'
atten_rate_filename = "atten_rate_calibrated.npz"

# load the calibrated attenuation rate and the flight mask
atten_rate = np.load(calibrated_folder + atten_rate_filename)
atten_rate_data = atten_rate['atten_rate_calibrated']
flight_mask     = atten_rate['flight_mask']

# load the parameter file for reverse standardization
preprocess_param = pd.read_csv(param_path, index_col=0)
Ns_standardize_epsilon = preprocess_param['Ns_standardization_epsilon'].values[0]
print("Ns_standardize_epsilon:", Ns_standardize_epsilon)

# load the standardization parameters for the training data
Ns_mean_data = scipy.io.loadmat(data_path + Ns_mean_filename)
Ns_std_data  = scipy.io.loadmat(data_path + Ns_std_filename)
Ns_mean = Ns_mean_data['Ns_mean_masked']
Ns_std = Ns_std_data['Ns_std_masked']  

# chemistry: this comes from matching the median of simulated and observed attenuation rate (processTrainingData.m)
Hp = 5.928571e-07 

# bring attenuation rate back to the physical dimension


Ns_standardize_epsilon: 0.262291780650704
